In [1]:
import sys
print(sys.prefix)

c:\Users\saffa\Desktop\rag-project\.venv


In [4]:
from pathlib import Path
DOCS_DIR = Path(r"C:\Users\saffa\Desktop\fastapi\docs\en\docs")
md_files = sorted(DOCS_DIR.rglob("*.md"))
print(len(md_files))

158


In [5]:
for path in md_files[:10]:
    print(path.relative_to(DOCS_DIR))

_llm-test.md
about\index.md
advanced\additional-responses.md
advanced\additional-status-codes.md
advanced\advanced-dependencies.md
advanced\advanced-python-types.md
advanced\async-tests.md
advanced\behind-a-proxy.md
advanced\custom-response.md
advanced\dataclasses.md


In [7]:
docs = []
for path in md_files:
    text = path.read_text(encoding="utf-8")
    source = path.relative_to(DOCS_DIR).as_posix()
    docs.append({"source": source, "text": text})

print(len(docs))
print(docs[1]["source"])

158
about/index.md


In [8]:
page = next(d for d in docs if d["source"] == "tutorial/first-steps.md")
print(page["text"][:1500])

# First Steps { #first-steps }

The simplest FastAPI file could look like this:

{* ../../docs_src/first_steps/tutorial001.py *}

Copy that to a file `main.py`.

/// tip

FastAPI has an [official extension for VS Code](https://marketplace.visualstudio.com/items?itemName=FastAPILabs.fastapi-vscode) (and Cursor), which provides a lot of features, including a path operation explorer, path operation search, CodeLens navigation in tests (jump to definition from tests), and FastAPI Cloud deployment and logs, all from your editor.

///

Run the live server:

<div class="termy">

```console
$ <font color="#4E9A06">uv run fastapi</font> dev

  <span style="background-color:#009485"><font color="#D3D7CF"> FastAPI </font></span>  Starting development server 🚀

             Searching for package file structure from directories
             with <font color="#3465A4">__init__.py</font> files
             Importing from <font color="#75507B">/home/user/code/</font><font color="#AD7FA8">awesomeapp</f

In [9]:
pages_with_includes = [d["source"] for d in docs if "{*" in d["text"]]
print(len(pages_with_includes))

91


In [10]:
import re

pattern = r"\{\*.*?\*\}"
all_includes = []

for d in docs:
    found = re.findall(pattern, d["text"])
    all_includes.extend(found)

print(len(all_includes))
for inc in all_includes[:15]:
    print(inc)

453
{* ../../docs_src/additional_responses/tutorial001.py hl[18,22] *}
{* ../../docs_src/additional_responses/tutorial002.py hl[17:22,26] *}
{* ../../docs_src/additional_responses/tutorial003.py hl[20:31] *}
{* ../../docs_src/additional_responses/tutorial004.py hl[11:15,24] *}
{* ../../docs_src/additional_status_codes/tutorial001.py hl[4,25] *}
{* ../../docs_src/dependencies/tutorial011.py hl[12] *}
{* ../../docs_src/dependencies/tutorial011.py hl[9] *}
{* ../../docs_src/dependencies/tutorial011.py hl[18] *}
{* ../../docs_src/dependencies/tutorial011.py hl[22] *}
{* ../../docs_src/dependencies/tutorial013.py *}
{* ../../docs_src/dependencies/tutorial013.py ln[19:21] *}
{* ../../docs_src/dependencies/tutorial013.py ln[30:38] hl[31:33] *}
{* ../../docs_src/dependencies/tutorial014.py ln[24:28] hl[28] *}
{* ../../docs_src/async_tests/app_a/main.py *}
{* ../../docs_src/async_tests/app_a/test_main.py *}


In [15]:
directive = "{* ../../docs_src/dependencies/tutorial013.py ln[30:38] hl[31:33] *}"

inner = directive[2:-2]
parts = inner.split()

print(parts)
path_str = parts[0]
options = parts[1:]

print(path_str)
print(options)

['../../docs_src/dependencies/tutorial013.py', 'ln[30:38]', 'hl[31:33]']
../../docs_src/dependencies/tutorial013.py
['ln[30:38]', 'hl[31:33]']


In [16]:
candidate_a = (DOCS_DIR / path_str).resolve()
candidate_b = (DOCS_DIR.parent / path_str).resolve()

print(candidate_a, candidate_a.exists())
print(candidate_b, candidate_b.exists())

C:\Users\saffa\Desktop\fastapi\docs\docs_src\dependencies\tutorial013.py False
C:\Users\saffa\Desktop\fastapi\docs_src\dependencies\tutorial013.py True


In [19]:
INCLUDE_BASE = DOCS_DIR.parent

missing = []

for directive in all_includes:
    parts = directive[2:-2]
    path_str = parts[0]
    full_path = (INCLUDE_BASE / path_str).resolve()
    if not full_path.exists():
        missing.append(full_path)

print(len(all_includes) - len(missing), "found")
print(len(missing), "missing")

453 found
0 missing


In [20]:
[d["source"] for d in docs if "tutorial013.py ln[30:38]" in d["text"]]

['advanced/advanced-dependencies.md']

## Finding: ln[a:b] format
- ln[a:b] = editor lines a to b, 1-based, inclusive
- Verified with tutorial013.py ln[30:38] against the website (also hl[31:33] highlights lines 31-33)
- Include paths are relative to docs/en (all 453 resolved)

In [21]:
code_path = INCLUDE_BASE / "../../docs_src/dependencies/tutorial013.py"
lines = code_path.resolve().read_text(encoding="utf-8").splitlines()
print("\n".join(lines[29:38]))

def generate_stream(query: str):
    for ch in query:
        yield ch
        time.sleep(0.1)


@app.get("/generate", dependencies=[Depends(get_user)])
def generate(query: str):
    return StreamingResponse(content=generate_stream(query))


In [23]:
opt = "ln[30:38]"
inside = opt[3:-1]          # get just "30:38"
start_str, end_str = inside.split(":")
start, end = int(start_str), int(end_str)
print(start, end)

30 38


In [28]:
option_kinds = set()

for directive in all_includes:
    parts = directive[2:-2].split()
    for opt in parts[1:]:                  # only the options, not the path
        kind = opt.split("[")[0]          # "ln[30:38]" → "ln"
        option_kinds.add(kind)              # sets use a different method than lists

print(option_kinds)

{'39:40]', '7:8]', '8:9]', '14:16]', 'title', '19:32]', '6:8]', 'ln', 'hl', '22]', '19]'}


In [29]:
weird = [d for d in all_includes if " 22]" in d or "title" in d]
for w in weird[:10]:
    print(w)

{* ../../docs_src/path_operation_advanced_configuration/tutorial007.py hl[15:20, 22] *}
{* ../../docs_src/bigger_applications/app/routers/users.py hl[1,3] title["app/routers/users.py"] *}
{* ../../docs_src/bigger_applications/app/routers/users.py hl[6,11,16] title["app/routers/users.py"] *}
{* ../../docs_src/bigger_applications/app/dependencies.py hl[3,6:8] title["app/dependencies.py"] *}
{* ../../docs_src/bigger_applications/app/routers/items.py hl[5:10,16,21] title["app/routers/items.py"] *}
{* ../../docs_src/bigger_applications/app/routers/items.py hl[3] title["app/routers/items.py"] *}
{* ../../docs_src/bigger_applications/app/routers/items.py hl[30:31] title["app/routers/items.py"] *}
{* ../../docs_src/bigger_applications/app/main.py hl[1,3,7] title["app/main.py"] *}
{* ../../docs_src/bigger_applications/app/main.py hl[4:5] title["app/main.py"] *}
{* ../../docs_src/bigger_applications/app/main.py hl[5] title["app/main.py"] *}


In [30]:
for frag in ["39:40]", "7:8]", "8:9]", "14:16]", "19:32]", "6:8]", "19]"]:
    for d in all_includes:
        if " " + frag in d:
            print(d)

{* ../../docs_src/path_operation_advanced_configuration/tutorial006.py hl[19:36, 39:40] *}
{* ../../docs_src/response_headers/tutorial002.py hl[1, 7:8] *}
{* ../../docs_src/response_cookies/tutorial002.py hl[1, 8:9] *}
{* ../../docs_src/sub_applications/tutorial001.py hl[11, 14:16] *}
{* ../../docs_src/metadata/tutorial001.py hl[3:16, 19:32] *}
{* ../../docs_src/sub_applications/tutorial001.py hl[3, 6:8] *}
{* ../../docs_src/sub_applications/tutorial001.py hl[11, 19] *}


In [34]:
directive = '{* ../../docs_src/x.py hl[15:20, 22] ln[30:38] title["app/main.py"] *}'

m = re.search(r"ln\[(.*?)\]", directive)
print(m)
print(m.group(1))

<re.Match object; span=(37, 46), match='ln[30:38]'>
30:38


In [32]:
m2 = re.search(r"ln\[(.*?)\]", "{* ../../docs_src/first_steps/tutorial001.py *}")
print(m2)

None


In [35]:
m2.group(1)

AttributeError: 'NoneType' object has no attribute 'group'

In [38]:
m = re.search(r"ln\[(.*?)\]", directive)
if m:                      # None is falsy, a match object is truthy
    print("ln found:", m.group(1))
else:
    print("no ln, use the whole file")

ln found: 30:38


In [40]:
ln_values = []

for directive in all_includes:
    m = re.search(r"ln\[(.*?)\]", directive)
    if m:
        ln_values.append(m.group(1))

print(len(ln_values), "placeholders have ln")

bad = []
for v in ln_values:
    parts = v.split(":")
    if len(parts) != 2 or not parts[0].isdigit() or not parts[1].isdigit():
        bad.append(v)

print(bad)

49 placeholders have ln
['1:9,29:35', '1:2,12:16,29,38:41', '1:2,19:26,29,44:46', '6,19:20']


In [44]:
def parse_ln(value):
    """Turn '1:2,12:16,29' into [(1, 2), (12, 16), (29, 29)]."""
    ranges = []
    for item in value.split(","):
        if ":" in item:
            start_str, end_str = item.split(":")
            ranges.append((int(start_str), int(end_str)))
        else:
            ranges.append((int(item), int(item)))
    return ranges


In [45]:
print(parse_ln("30:38"))
print(parse_ln("1:2,12:16,29,38:41"))
print(parse_ln("6,19:20"))

[(30, 38)]
[(1, 2), (12, 16), (29, 29), (38, 41)]
[(6, 6), (19, 20)]


In [46]:
bad = []
for v in ln_values:
    for start, end in parse_ln(v):
        if start > end:
            bad.append(v)

print("parsed all", len(ln_values))
print(bad)

parsed all 49
[]


In [47]:
def select_lines(code_lines, ranges):
    """Keep only the given (start, end) line ranges (1-based, inclusive), with '# ...' between them."""
    pieces = []
    for start, end in ranges:
        chunk = code_lines[start-1:end]
        pieces.append("\n".join(chunk))
    return "\n# ...\n".join(pieces)

In [48]:
code_path = (INCLUDE_BASE / "../../docs_src/dependencies/tutorial013.py").resolve()
code_lines = code_path.read_text(encoding="utf-8").splitlines()

print(select_lines(code_lines, [(30, 38)]))
print("=" * 40)
print(select_lines(code_lines, [(1, 2), (30, 31)]))

def generate_stream(query: str):
    for ch in query:
        yield ch
        time.sleep(0.1)


@app.get("/generate", dependencies=[Depends(get_user)])
def generate(query: str):
    return StreamingResponse(content=generate_stream(query))
import time
from typing import Annotated
# ...
def generate_stream(query: str):
    for ch in query:


## Session 3 findings
- Corpus: FastAPI English docs (docs/en/docs), 158 pages, MIT licence
- 91 pages contain {* ... *} code placeholders, 453 in total
- Placeholder paths are relative to docs/en (all 453 resolve)
- Options: ln (lines to show), hl (highlight only, ignored), title (filename, keep)
- ln[...] = 1-based, inclusive; can hold several ranges and single lines, e.g. ln[1:2,12:16,29]
- Splitting on spaces broke on hl[15:20, 22], so extract fields with regex instead
- FastAPI hash: a1b3be01a